# 02 — Estrategia de búsqueda reproducible

Este notebook construye consultas multilingües para la revisión integral sobre cambio climático y pesquerías.

Se mantienen dos archivos distintos:

1. `planned_queries.csv`: consultas conceptuales generadas automáticamente;
2. `search_log_working.csv`: búsquedas realmente ejecutadas en una plataforma o sitio institucional.

La existencia de una consulta planificada no implica que la búsqueda haya sido ejecutada.


In [ ]:
from pathlib import Path
import shutil
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from evidence_review.search import (
    load_search_strategy,
    build_standard_queries,
    standard_queries_to_records,
)

print(f'Project root: {ROOT}')


## 1. Cargar la estrategia


In [ ]:
strategy_path = ROOT / 'config' / 'search_strategy.yml'
strategy = load_search_strategy(strategy_path)

print(strategy['project']['review_title'])
print('Languages:', ', '.join(strategy['eligibility']['languages']))
print('Date from:', strategy['eligibility']['date_from'])


## 2. Generar y exportar las consultas planificadas


In [ ]:
queries = build_standard_queries(strategy)
queries_df = pd.DataFrame(standard_queries_to_records(queries))

planned_path = ROOT / 'data' / 'interim' / 'planned_queries.csv'
planned_path.parent.mkdir(parents=True, exist_ok=True)
queries_df.to_csv(planned_path, index=False, encoding='utf-8-sig')

print(f'Planned queries: {len(queries_df)}')
print(f'Saved: {planned_path.relative_to(ROOT)}')
queries_df[['query_id', 'language', 'query_type', 'blocks']]


## 3. Inspeccionar una consulta completa

Antes de utilizar una consulta en Scopus, Web of Science, OpenAlex u otra plataforma,
debe adaptarse a la sintaxis específica de esa plataforma. La consulta finalmente
ejecutada se registra exactamente en el archivo de búsquedas.


In [ ]:
selected_language = 'en'
selected_type = 'priority_taxa'

selected = queries_df.query(
    'language == @selected_language and query_type == @selected_type'
).iloc[0]

print(selected['exact_query'])


## 4. Preparar el registro de búsquedas ejecutadas

`search_log_working.csv` queda vacío hasta que una búsqueda se ejecute realmente.
Cada fila debe documentar plataforma, fecha, consulta exacta, filtros, resultados y
archivo exportado.


In [ ]:
template_path = ROOT / 'data' / 'templates' / 'search_log.csv'
working_path = ROOT / 'data' / 'interim' / 'search_log_working.csv'
working_path.parent.mkdir(parents=True, exist_ok=True)

if not working_path.exists():
    shutil.copyfile(template_path, working_path)
    print(f'Created: {working_path.relative_to(ROOT)}')
else:
    print(f'Already exists: {working_path.relative_to(ROOT)}')

search_log = pd.read_csv(working_path)
print(f'Executed searches recorded: {len(search_log)}')
search_log.head()


## 5. Preparar una fila candidata para una búsqueda real


In [ ]:
candidate_search = {
    'search_id': 'YYYYMMDD_platform_language_querytype',
    'search_date': '',
    'reviewer': '',
    'source_channel': 'scientific_database',
    'platform_or_website': '',
    'database_collection': '',
    'language': selected_language,
    'query_type': selected_type,
    'exact_query': selected['exact_query'],
    'filters_applied': '',
    'date_from': strategy['eligibility']['date_from'],
    'date_to': strategy['eligibility']['date_to'],
    'result_count': None,
    'export_filename': '',
    'export_format': '',
    'search_status': 'planned',
    'notes': '',
}

pd.DataFrame([candidate_search])


## 6. Fuentes prioritarias


In [ ]:
scientific = pd.DataFrame(strategy['search_channels']['scientific_databases'])
institutional = pd.DataFrame({
    'institutional_source': strategy['search_channels']['institutional_sources']
})

display(scientific)
display(institutional)


## Criterio para avanzar

Antes de importar resultados:

- revisar las ocho consultas planificadas;
- seleccionar una plataforma piloto;
- adaptar la sintaxis sin cambiar el significado;
- ejecutar la búsqueda;
- registrar una fila en `search_log_working.csv`;
- exportar los resultados sin modificar el archivo original;
- comenzar con una prueba piloto antes de la búsqueda completa.
